This notebook contains the initial processing and exploration of the benchmark results.

In [1]:
import profilelib.*
%use dataframe


In [2]:
@DataSchema
data class BenchmarkRow(
    val benchmark: String,
    val jvm: Double,
    val jvmError: Double,
    val native: Double,
    val nativeError: Double
)

fun loadAsDF(path: String): DataFrame<BenchmarkRow> {
    val jvm = loadScores(loadJson(dataPath(path.replace("*", "jvm"))))
    val native = loadScores(loadJson(dataPath(path.replace("*", "linuxX64"))))
    if (jvm.keys != native.keys) throw Exception()
    val benchmarks = jvm.keys.toList()

    fun Map<String, Double>.toNamedColumn(name: String): DataColumn<Double> = benchmarks
        .map { this[it]!! }
        .toColumn(name)

    return dataFrameOf(
        benchmarks.toColumn("benchmark"),
        jvm.mapValues { it.value.first }.toNamedColumn("jvm"),
        jvm.mapValues { it.value.second }.toNamedColumn("jvmError"),
        native.mapValues { it.value.first }.toNamedColumn("native"),
        native.mapValues { it.value.second }.toNamedColumn("nativeError"),
    ) as DataFrame<BenchmarkRow>
}

In [3]:
// Workaround for dataFrameConfig.display.decimalFormat = RendererDecimalFormat.of("%.2e")
// Confirmed by Kotlin Notebook that this doesn't and shouldn't work

fun DataFrame<*>.display() = convert { colsAtAnyDepth().colsOf<Double>() }.with { "%.2e".format(it) }

In [4]:
// f593cea Map benchmark

loadAsDF("benchmarks/report-map-*.json")
    .add("ratio") { native / jvm }

benchmark,jvm,jvmError,native,nativeError,ratio
"MapBenchmark.controlB{size=""10""}",759119857.326849,2139402.725089,45320236.612710,207877.330196,0.059701
"MapBenchmark.controlB{size=""50""}",762446547.156493,18687427.474287,44668290.257533,193013.894885,0.058585
"MapBenchmark.controlB{size=""100""}",763925819.198096,34628628.802514,45450557.872337,72973.889414,0.059496
"MapBenchmark.controlB{size=""500""}",767693666.829037,29713713.731092,45286895.911187,73364.443041,0.058991
"MapBenchmark.controlB{size=""1000""}",757783143.131827,23556969.276731,44695049.715614,112422.409830,0.058981
"MapBenchmark.controlB{size=""5000""}",767072141.019164,29006489.952450,44836739.441361,381563.612162,0.058452
"MapBenchmark.controlB{size=""10000""}",748218820.942364,20046888.607717,46863722.060992,79497.341395,0.062634
"MapBenchmark.controlB{size=""50000""}",761032975.655372,11699539.865571,46695816.083055,62971.424581,0.061358
"MapBenchmark.controlB{size=""100000""}",749865409.047599,4897603.677021,46687021.726093,44141.337202,0.062261
"MapBenchmark.controlF{size=""10""}",746372402.644470,11452266.755355,45220468.578308,30572.529146,0.060587


In [5]:
// [master d719de9] Fix benchmarks for StringBuilder
loadAsDF("benchmarks/report-stringbuilder-*.json")
    .add("ratio") { native / jvm }

benchmark,jvm,jvmError,native,nativeError,ratio
StringBuilderBenchmark.appendBig,2011.418537,10.703042,251.584028,2.408974,0.125078
StringBuilderBenchmark.appendMedium,1190.032920,13.736790,197.422228,1.986424,0.165896
StringBuilderBenchmark.appendSmall,191.171800,1.230805,19.300317,0.060918,0.100958


Let's do it with avgt instead.

In [6]:
fun findControlName(bmName: String): String = when (bmName.substringBefore(".")) {
    "MapBenchmark" -> bmName.takeIf { it.contains("get") }?.replace(Regex("get(.)\\d*")) { "control${it.groupValues[1]}" } ?: bmName
    "StringSubstringBenchmark" -> bmName.substringBefore("_") + "_control"
    "StringBuilderBenchmark" -> "StringBuilderBenchmark.control"
    else -> throw Exception()
}

// 2c1a32c Fix Map benchmarks

val df = loadAsDF("benchmarks/report-avgt-*.json")
    .add("jvmControl") { df().single{ benchmark == findControlName(this@add.benchmark) }.jvm }
    .add("jvmControlError") { df().single{ benchmark == findControlName(this@add.benchmark) }.jvmError }
    .add("nativeControl") { df().single{ benchmark == findControlName(this@add.benchmark) }.native }
    .add("nativeControlError") { df().single{ benchmark == findControlName(this@add.benchmark) }.nativeError }
    .also { notebook.display(it.display()) }
    .filter { !benchmark.contains("control") }

benchmark,jvm,jvmError,native,nativeError,jvmControl,jvmControlError,nativeControl,nativeControlError
"MapBenchmark.controlB{size=""10""}",1.38e-09,1.57e-10,2.31e-08,1.01e-10,1.38e-09,1.57e-10,2.31e-08,1.01e-10
"MapBenchmark.controlB{size=""50""}",1.35e-09,1.51e-10,2.30e-08,5.20e-11,1.35e-09,1.51e-10,2.30e-08,5.20e-11
"MapBenchmark.controlB{size=""100""}",1.34e-09,1.26e-12,2.46e-08,1.13e-09,1.34e-09,1.26e-12,2.46e-08,1.13e-09
"MapBenchmark.controlB{size=""500""}",1.33e-09,1.30e-11,2.52e-08,7.25e-11,1.33e-09,1.30e-11,2.52e-08,7.25e-11
"MapBenchmark.controlB{size=""1000""}",1.34e-09,5.53e-12,2.29e-08,3.17e-11,1.34e-09,5.53e-12,2.29e-08,3.17e-11
"MapBenchmark.controlB{size=""5000""}",1.35e-09,8.74e-12,2.48e-08,8.33e-11,1.35e-09,8.74e-12,2.48e-08,8.33e-11
"MapBenchmark.controlB{size=""10000""}",1.35e-09,9.84e-11,2.31e-08,3.27e-11,1.35e-09,9.84e-11,2.31e-08,3.27e-11
"MapBenchmark.controlB{size=""50000""}",1.35e-09,8.59e-11,2.30e-08,6.04e-11,1.35e-09,8.59e-11,2.30e-08,6.04e-11
"MapBenchmark.controlB{size=""100000""}",1.39e-09,1.68e-10,2.30e-08,9.88e-11,1.39e-09,1.68e-10,2.30e-08,9.88e-11
"MapBenchmark.controlF{size=""10""}",1.41e-09,6.62e-12,2.36e-08,3.25e-11,1.41e-09,6.62e-12,2.36e-08,3.25e-11


In [7]:
val avgtDF = df
    .add("naiveRatio") { jvm / native }
    .add("naiveRatioError") { propagateErrorThroughRatio(jvm, jvmError, native, nativeError) }
    .add("adjustedRatio") { (jvm - jvmControl) / (native - nativeControl) }
    .add("adjustedRatioError") { propagateErrorThroughRatio(
        jvm - jvmControl,
        propagateErrorThroughSum(jvmError, jvmControlError),
        native - nativeControl,
        propagateErrorThroughSum(nativeError, nativeControlError),
    )}
    .remove { cols { (it.name() in df.columns().map { it.name }) && it.name() != "benchmark" } }
avgtDF
    .convert("naiveRatioError") { "%.2e".format(it) }
    .convert("adjustedRatioError") { "%.2e".format(it) }


benchmark,naiveRatio,naiveRatioError,adjustedRatio,adjustedRatioError
"MapBenchmark.getB2{size=""10""}",0.052738,1.51e-07,0.049684,7.12e-07
"MapBenchmark.getB2{size=""50""}",0.048475,9.02e-07,0.044486,1.23e-06
"MapBenchmark.getB2{size=""100""}",0.069339,1.64e-07,0.074148,1.13e-06
"MapBenchmark.getB2{size=""500""}",0.053153,1.06e-06,0.053301,1.31e-06
"MapBenchmark.getB2{size=""1000""}",0.070003,1.08e-07,0.073817,1.32e-07
"MapBenchmark.getB2{size=""5000""}",0.049409,2.28e-07,0.047301,2.86e-07
"MapBenchmark.getB2{size=""10000""}",0.076229,3.53e-06,0.082070,4.08e-06
"MapBenchmark.getB2{size=""50000""}",0.047334,4.31e-07,0.042935,6.21e-07
"MapBenchmark.getB2{size=""100000""}",0.054017,3.66e-07,0.051131,8.61e-07
"MapBenchmark.getB5{size=""10""}",0.047759,1.39e-06,0.043004,1.77e-06


In [8]:
data class JmhResult (
    val name: String,
    val score: Double,
    val error: Double,
    )

val df = loadScores(loadJson(dataPath("benchmarks/jvm-Map.json")))
    .map { JmhResult(it.key, it.value.first, it.value.second) }
    .toDataFrame()
    .also { notebook.display(it.display()) }

name,score,error
"MapBenchmark.konst{miss=""true"",size=""...",3.31e-09,3.80e-11
"MapBenchmark.konst{miss=""true"",size=""...",3.32e-09,3.25e-11
"MapBenchmark.konst{miss=""true"",size=""...",3.39e-09,6.07e-11
"MapBenchmark.konst{miss=""true"",size=""...",3.37e-09,1.68e-11
"MapBenchmark.konst{miss=""false"",size=...",3.35e-09,3.15e-11
"MapBenchmark.konst{miss=""false"",size=...",3.37e-09,4.19e-11
"MapBenchmark.konst{miss=""false"",size=...",3.33e-09,4.09e-11
"MapBenchmark.konst{miss=""false"",size=...",3.36e-09,4.58e-11
"MapBenchmark.konst_control{miss=""true...",3.01e-10,6.96e-12
"MapBenchmark.konst_control{miss=""true...",3.01e-10,8.02e-12


In [9]:
df
    .filter { !name.contains("control") }
    .remove { error }
    .add("control") { df.single { name == this@add.name.replace("{", "_control{") }.score }
    .add("adjusted") { score - this["control"] as Double }
    .let { both ->
        both.filter { name.contains("konst") }.joinWith(both.filter { name.contains("random") }) {
            right.name.substringAfter("random") == name.substringAfter("konst")
        }
    }
    .rename { this["adjusted"] }.into("konst")
    .rename { this["adjusted1"] }.into("random")
    .add("params") { name.substringAfter("{").dropLast(1) }
    .select { this["params"] and this["konst"] and this["random"] }
    .add("k") { "%.1f".format(this["random"] as Double / this["konst"] as Double) }
    .display()

params,konst,random,k
"miss=""true"",size=""127""",3.00e-09,3.49e-09,1.2
"miss=""true"",size=""1023""",3.02e-09,3.07e-09,1.0
"miss=""true"",size=""8191""",3.09e-09,3.36e-09,1.1
"miss=""true"",size=""131071""",3.08e-09,1.63e-08,5.3
"miss=""false"",size=""127""",3.05e-09,3.22e-09,1.1
"miss=""false"",size=""1023""",3.06e-09,3.56e-09,1.2
"miss=""false"",size=""8191""",3.04e-09,3.37e-09,1.1
"miss=""false"",size=""131071""",3.06e-09,8.57e-09,2.8


In [10]:
fun findControlName(name: String) = name.takeIf { it.contains("_") } ?: name.replace("{", "_control{")

val mapDF = loadAsDF("benchmarks/report-randommap-*.json")
    .add("jvmControl") { df().single{ benchmark == findControlName(this@add.benchmark) }.jvm }
    .add("jvmControlError") { df().single{ benchmark == findControlName(this@add.benchmark) }.jvmError }
    .add("nativeControl") { df().single{ benchmark == findControlName(this@add.benchmark) }.native }
    .add("nativeControlError") { df().single{ benchmark == findControlName(this@add.benchmark) }.nativeError }
    .filter { !benchmark.contains("control") }
    .add("ratio") { (jvm - this["jvmControl"] as Double) / (native - this["nativeControl"] as Double) }

mapDF.convert("ratio") { "%.2f".format(it) }.display()

benchmark,jvm,jvmError,native,nativeError,jvmControl,jvmControlError,nativeControl,nativeControlError,ratio
"RandomMapBenchmark.konst{miss=""true"",...",3.93e-09,4.55e-10,8.01e-08,1.60e-10,1.37e-09,2.02e-10,2.18e-08,7.15e-11,0.04
"RandomMapBenchmark.konst{miss=""true"",...",3.91e-09,5.01e-10,7.99e-08,1.68e-10,1.35e-09,1.92e-10,2.19e-08,6.04e-11,0.04
"RandomMapBenchmark.konst{miss=""true"",...",4.04e-09,3.03e-11,7.98e-08,6.00e-11,1.38e-09,2.16e-10,2.19e-08,6.36e-11,0.05
"RandomMapBenchmark.konst{miss=""true"",...",4.00e-09,2.69e-11,7.71e-08,2.30e-09,1.32e-09,1.01e-11,2.41e-08,3.89e-11,0.05
"RandomMapBenchmark.konst{miss=""false""...",3.98e-09,3.24e-10,8.00e-08,1.65e-10,1.42e-09,2.41e-11,2.17e-08,3.21e-11,0.04
"RandomMapBenchmark.konst{miss=""false""...",3.89e-09,4.18e-10,7.38e-08,1.77e-10,1.34e-09,1.85e-10,2.42e-08,9.26e-11,0.05
"RandomMapBenchmark.konst{miss=""false""...",3.78e-09,1.30e-10,7.69e-08,2.59e-09,1.32e-09,1.73e-11,2.16e-08,7.92e-11,0.04
"RandomMapBenchmark.konst{miss=""false""...",3.98e-09,6.13e-10,8.04e-08,1.76e-10,1.32e-09,4.26e-12,2.36e-08,8.17e-11,0.05
"RandomMapBenchmark.random{miss=""true""...",5.63e-09,7.91e-10,8.00e-08,9.15e-11,1.85e-09,2.80e-10,3.47e-08,1.94e-10,0.08
"RandomMapBenchmark.random{miss=""true""...",4.58e-09,1.15e-11,7.67e-08,5.27e-11,1.92e-09,2.54e-10,3.46e-08,1.25e-10,0.06


In [11]:
import org.jetbrains.kotlin.psi.addRemoveModifier.addAnnotationEntry

plot {
    points {
        buildList {
            avgtDF.filter { benchmark.contains("StringBuilder") }.forEach {
                add("StringBuilder" to adjustedRatio)
            }
            add("StringSubstring" to 0.1898929936770031)
            add("StringSubstring" to 0.2059796100901835)
            mapDF.forEach {
                add("Map" to ratio)
            }
        }.let {
            x(it.map { it.first })
            y(it.map { it.second })
        }
    }
    points {
        listOf(
            "StringBuilder" to 0.154,
            "Map" to 0.129,
            "StringSubstring" to 0.174,
        ).let {
            x(it.map { it.first })
            y(it.map { it.second })
        }
        symbol = Symbol.CROSS
        size = 10.0
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="XtId6t"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("XtId6t");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["StringBuilder","StringBuilder","StringBuilder","StringSubstring","StringSubstring","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map"],
"y":[0.12032088028666876,0.16017303786785733,0.10015870921982577,0.1898929936770031,0.2059796100901835,0.04391425355183799,0.044160435008388295,0.04595967423958263,0.05049533654307476,0.04398849426005149,0.051398868486241096,0.04460080453064427,0.046792905563907417,0.08358120840546561,0.06305326283115797,0.09622949870475024,0.26878017203769156,0.05490087817326117,0.07453074004780358,0.07076331789763465,0.14544982907381857]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["StringBuilder","Map","StringSubstring"],
"y":[0.154,0.129,0.174]
},
"shape":4.0,
"size":10.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"2"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 StringBuilder 
 
 
 
 
 
 
 
 
 StringSubstring 
 
 
 
 
 
 
 
 
 Map 
 
 
 
 
 
 
 
 
 
 
 0.05 
 
 
 
 
 
 
 0.1 
 
 
 
 
 
 
 0.15 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.25 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x